# 1. Costed fire stations (minimum-cost set cover)

Lecture: *Week 6 - Integer Linear Programming - Further Problems*, PDF pages 3–8. The first of three versions
of the same model: here every constraint is typed out by hand, exactly as on page 7.

**The problem.** Four fire stations $a, b, c, d$ can be run; each covers some of the cities 0–4 and has a
running cost (slides, PDF pages 6–8):

| Station | Covers cities | Cost |
| --- | --- | --- |
| $a$ | 0, 2 | 12 |
| $b$ | 1, 2, 3 | 7 |
| $c$ | 1, 4 | 10 |
| $d$ | 4 | 5 |

Choose stations so that every city is covered by at least one of them, at the lowest total cost. $x_t = 1$ if
station $t$ is run:

minimise $\sum_{t \in S} c_t x_t$ subject to $\sum_{t:\, e \in s_t} x_t \ge 1$ for every city $e$, $x_t \in \{0, 1\}$.

It is the minimum set cover of Week 05 (demo 3), with a cost on each set.

In the code the stations $a, b, c, d$ are `x[0]` … `x[3]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Cost **24**: stations $a$, $b$ and $d$ (`x[0]`, `x[1]`, `x[3]` = 1), as on page 8.

*Original script: `5 Source Files/Python/costed_firestation.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

Create the model, one binary variable per station, and the list of running costs.

In [ ]:
# Create the model and one binary variable per station: x[0]..x[3] = stations a..d
model = ConcreteModel()

model.x = Var(range(4), within=Binary)

c = [12, 7, 10, 5]   # running cost of stations a, b, c, d

The objective: the total running cost $12x_a + 7x_b + 10x_c + 5x_d$, minimised.

In [ ]:
# Create the optimization problem
model.obj = Objective(expr=sum(c[i] * model.x[i] for i in model.x), sense=minimize)

One constraint per city: at least one of the stations that cover it must be run (page 7, right).

In [ ]:
# Define the inequality constraints: one per city, "covered at least once"
model.city0 = Constraint(expr=model.x[0] >= 1)                 # city 0: only station a covers it
model.city1 = Constraint(expr=model.x[1] + model.x[2] >= 1)    # city 1: stations b or c
model.city2 = Constraint(expr=model.x[0] + model.x[1] >= 1)    # city 2: stations a or b
model.city3 = Constraint(expr=model.x[1] >= 1)                 # city 3: only station b
model.city4 = Constraint(expr=model.x[2] + model.x[3] >= 1)    # city 4: stations c or d

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model with the data filled in: compare it with page 7.

In [ ]:
# Display the model
model.pprint()

Solve. In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

`display()` prints the chosen stations (value 1) and the objective (the total cost).

In [ ]:
# Display the results
model.display()

## What do you notice?

Two of the four decisions are made by the constraints alone. Which ones, and what is left for the solver to
choose?

<details><summary>Answer</summary>

City 0 is covered only by station $a$ and city 3 only by station $b$, so $x_a = x_b = 1$ is forced. Together
they cover cities 0–3. City 4 still needs station $c$ (cost 10) or $d$ (cost 5), so the solver picks $d$:
$12 + 7 + 5 = 24$. No other choice of stations is as cheap.
</details>